# S4 J4 — Memory Layer — Teacher

Semaine 4 — Jour 4 : Memory Layer

## Objectifs

- Concevoir une couche mémoire d'agents.
- Implémenter un store in-memory testable.
- Filtrer par namespace, visibilité et TTL.
- Générer un context pack injectible.

## Rappel conceptuel

La mémoire durable ne doit pas être confondue avec l'historique conversationnel. Elle conserve uniquement les informations réutilisables, gouvernées et effaçables.

In [ ]:
from pathlib import Path
import sys

# Adapter le chemin si le notebook est déplacé.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "mini_framework").exists():
    REPO_ROOT = Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mini_framework.memory import MemoryStore, MemoryQuery

In [ ]:
from datetime import datetime, timezone

store = MemoryStore()
now = datetime(2026, 1, 1, 12, 0, tzinfo=timezone.utc)

store.add(
    namespace="user:demo",
    owner_agent="planner",
    kind="preference",
    content="L'utilisateur préfère les explications étape par étape.",
    visibility="private",
    tags=["preference", "format"],
    importance=0.9,
    now=now,
)

results = store.retrieve(
    MemoryQuery(namespace="user:demo", text="format étape", requester_agent="planner")
)

[(result.record.kind, result.record.content, result.score, result.reasons) for result in results]

## Exercice notebook

Ajoutez une mémoire `fact` partagée, puis récupérez uniquement les mémoires `shared` et `public`.

In [ ]:
# À compléter par l'apprenant

## Corrigés formateur

# Corrigé — Exercices

## Exercice 1

1. "L'utilisateur préfère les réponses en tableau." → `preference`
2. "Le paiement a échoué avec le code 402." → `tool_observation`
3. "La tâche courante attend une validation humaine." → `task_state`
4. "Ne jamais appeler l'outil delete sans confirmation." → `policy`
5. "User: peux-tu continuer ?" → `conversation`
6. "Le projet utilise un mini-framework maison." → `fact`

## Exercice 2

```json
{
  "namespace": "user:42",
  "owner_agent": "planner",
  "kind": "preference",
  "content": "L'utilisateur préfère recevoir les explications sous forme d'étapes numérotées.",
  "visibility": "private",
  "tags": ["preference", "format", "steps"],
  "importance": 0.9
}
```

## Exercice 3

L'agent non propriétaire peut recevoir :

```json
[
  {"content": "Fait partagé", "visibility": "shared"},
  {"content": "Annonce publique", "visibility": "public"}
]
```

La mémoire privée ne doit pas être injectée.

## Exercice 4

Un `task_state` est souvent temporaire : formulaire en cours, étape d'un workflow, validation attendue. Un TTL évite de reprendre un état périmé.

Une préférence utilisateur peut rester valable longtemps, mais elle doit quand même pouvoir être modifiée ou supprimée.

## Exercice 5

Exemple :

```text
score = importance
      + 2.0 * lexical_overlap_ratio
      + 1.2 * number_of_matching_tags
```

Chaque résultat doit aussi retourner des raisons : importance, mots communs, tags communs.

## Exercice 6

`forget_namespace("user:42")` doit supprimer toutes les mémoires de ce namespace, sans supprimer les autres namespaces. L'opération doit être auditée.

## Exercice 7

Tests indispensables :

1. isolation par namespace ;
2. respect de la visibilité ;
3. exclusion des records expirés ;
4. redaction PII ;
5. snapshot/restore ;
6. oubli namespace ;
7. scoring/ranking ;
8. audit log.

# Corrigé — Questions d'entretien

## Réponse 1

La mémoire conversationnelle sert à maintenir la continuité immédiate d'un dialogue. La mémoire durable conserve des informations réutilisables dans le futur, comme des préférences ou des faits vérifiés.

## Réponse 2

Le namespace empêche les fuites entre utilisateurs, tenants, équipes ou sessions. Sans namespace, une mémoire peut être réutilisée dans le mauvais contexte.

## Réponse 3

Toutes les mémoires ne sont pas pertinentes, autorisées ou à jour. L'injection automatique augmente le bruit, le coût, le risque de fuite et le risque d'instructions contradictoires.

## Réponse 4

Il faut identifier le namespace de l'utilisateur, supprimer toutes les mémoires associées, auditer l'opération et confirmer l'exécution. Les logs réglementaires éventuels doivent suivre la politique de conformité du produit.

## Réponse 5

Une préférence exprime une manière souhaitée d'interagir. Un fait vérifié décrit une information validée. Les deux ne doivent pas avoir la même durée de vie ni la même politique de partage.

## Réponse 6

La visibilité indique qui peut lire ou injecter la mémoire. Elle évite qu'une information privée soit partagée avec un agent ou un contexte non autorisé.

## Réponse 7

Le runner doit dépendre d'une interface `MemoryStore`, pas d'une classe concrète. PostgreSQL ou Redis deviennent alors des implémentations de cette interface.

## Réponse 8

Risques principaux : fuite inter-utilisateurs, stockage de PII, rétention excessive, réutilisation de données périmées, prompt injection persistante, confusion entre observation et fait.

## Réponse 9

Chaque écriture doit produire un événement : action, record, namespace, agent, version, timestamp. Une trace doit expliquer pourquoi une mémoire a été promue.

## Réponse 10

Une base vectorielle devient utile lorsque les mémoires sont nombreuses et que la recherche lexicalement exacte ne suffit plus. Elle doit rester derrière le contrat de la Memory Layer.

# Corrigé — Challenge

## Solution attendue

La solution fournie dans le lab implémente :

- `MemoryRecord`
- `MemoryQuery`
- `MemorySearchResult`
- `MemoryStore`
- `redact_pii`
- `promote_event`
- `snapshot`
- `from_snapshot`

## Points clés

### Isolation

Chaque record appartient à un namespace. `retrieve` et `forget_namespace` filtrent strictement sur ce namespace.

### Visibilité

Le champ `allowed_visibility` de `MemoryQuery` contrôle les records injectables.

### Expiration

Les records expirés sont exclus par défaut. Ils restent consultables uniquement si `include_expired=True`.

### Audit

Chaque `add`, `update`, `delete` et `forget` écrit un événement dans `audit_log`.

### Snapshot

`snapshot()` produit une structure JSON sérialisable. `from_snapshot()` restaure un store équivalent.

## Exemple

```python
store = MemoryStore()
store.add(
    namespace="user:42",
    owner_agent="planner",
    kind="preference",
    content="L'utilisateur préfère les étapes numérotées.",
    visibility="private",
    tags=["preference", "format"],
    importance=0.9,
)

results = store.retrieve(
    MemoryQuery(
        namespace="user:42",
        text="format étapes",
        allowed_visibility=("private", "shared"),
    )
)
```

## Vérification

Les tests du lab valident les critères d'acceptation du challenge.

# Review formateur

## Objectifs validés

L'apprenant doit être capable d'expliquer et de coder :

- un record mémoire ;
- une requête mémoire ;
- la séparation namespace/visibility/kind ;
- l'expiration TTL ;
- la redaction PII ;
- la restauration depuis snapshot ;
- l'audit des écritures.

## Points d'attention

### 1. Ne pas confondre mémoire et historique

Une erreur fréquente consiste à stocker chaque message comme mémoire durable. Le formateur doit insister : toute conversation n'est pas une mémoire.

### 2. Ne pas confondre recherche et injection

Retrouver une mémoire ne signifie pas qu'elle doit être injectée dans le prompt. Il faut encore filtrer, ranker, compacter et respecter le budget.

### 3. Ne pas créer de mémoire globale

Le namespace est obligatoire. Toute mémoire sans namespace est une dette de sécurité.

### 4. Éviter la persistance prématurée

Le lab utilise l'in-memory pour enseigner le contrat. La persistance arrive plus tard, mais le contrat doit déjà permettre le remplacement.

## Checklist de correction

- [ ] Le code s'exécute sans dépendance externe.
- [ ] Les tests passent.
- [ ] Les records ont un namespace.
- [ ] La visibilité est appliquée.
- [ ] Les records expirés sont exclus.
- [ ] `forget_namespace` ne supprime pas les autres namespaces.
- [ ] Le snapshot est restaurable.
- [ ] Les événements sont audités.
- [ ] Le code reste compréhensible pour un AI Engineer junior/intermédiaire.

## Propositions d'amélioration

- Ajouter une interface abstraite `MemoryBackend`.
- Ajouter une stratégie de compaction.
- Ajouter un backend SQLite.
- Ajouter une étape de validation humaine avant promotion de mémoire sensible.
- Ajouter un score hybride lexical + vectoriel lors de la semaine Knowledge Systems.

In [ ]:
store.add(
    namespace="user:demo",
    owner_agent="researcher",
    kind="fact",
    content="Le mini-framework expose une Memory Layer indépendante du backend.",
    visibility="shared",
    tags=["framework", "memory"],
    importance=0.8,
    now=now,
)

shared_results = store.retrieve(
    MemoryQuery(
        namespace="user:demo",
        text="framework memory",
        allowed_visibility=("shared", "public"),
        requester_agent="reviewer",
    )
)

[(result.record.visibility, result.record.content) for result in shared_results]